# Figures Generator

Runs the algorithm and then generates the figures used for the manuscript.

## Data and figures are not distributed

The simulation results (`notebooks/data/experiments_results.jld2`, several hundred MB) and the generated figures (`figures/`) are **not included in the public repository** because of their size. Both are fully regenerated by running this notebook from the `notebooks/` directory: the *Experiment 1* / *Experiment 2* cells create the `.jld2` file and the figure cells write the PNGs. Run settings (trials, generations, population size) are defined in the run-settings cell; shrink them for a quick smoke test and use the full values to reproduce the manuscript. See the repository `README.md` for setup instructions.

## Setup


In [ ]:
# Packages
display("text/html", "<style>.container { width:100% !important; }</style>")

# Julia dependencies for plotting, stats, and experiments
using Plots, LaTeXStrings, Distributions, DoubleFloats, Plots.PlotMeasures, LinearAlgebra,
    DataFrames, CSV, LsqFit, StatsBase, Distances, Parameters, Random, StatsBase, Graphs, 
    GraphPlot, Base.Threads, Printf, JLD2

using ProgressMeter
using Base.Threads: Atomic, atomic_add!

# Local modules
include("../src/wagner_algorithm.jl")
using ..BooleanNetwork: SimulationParameters, SimulationData
using ..BooleanNetwork: run_simulation
include("../src/data_processing.jl")
using ..CustomStats: summarize_history, summarize_simulation_run, alignment_score, 
    compute_mut_robustness
include("../scripts/run_experiments.jl")
using ..ExperimentHandler: Experiment1Config, Experiment2Config, Experiment1Result, 
    Experiment2Result, MotifStats
using ..ExperimentHandler: run_experiment_1, run_experiment_2, build_noise_scenarios

# Python bridge for motif search utilities
using PyCall
py"""
import sys
import os
sys.path.append(os.path.abspath("../src/"))
from motif_search import *
"""

In [ ]:
# Plotting utilities
const ROOT_DIR = normpath(joinpath(pwd(), ".."))
const FIGURES_DIR = joinpath(ROOT_DIR, "figures")
mkpath(FIGURES_DIR)
# Layout: figures/exp_1/<dataset_key>/... and figures/exp_2/<dataset_key>/...
# Default robustness folder for the single-dataset (standard) cells below.
const ROBUSTNESS_FIG_DIR = joinpath(FIGURES_DIR, "exp_1", "exp_1_standard", "scatterplots_mutational_robustness")
mkpath(ROBUSTNESS_FIG_DIR)

default(dpi=900, legendfontsize=24, guidefont=font(24), tickfont=font(24), margin=8Plots.mm, grid=false)
Plots.theme(:dao)
"""
    save_plot(plot_obj, filename)

Save a figure to the figures directory and print the output path.
"""
function save_plot(plot_obj, filename)
    filepath = joinpath(FIGURES_DIR, filename)
    mkpath(dirname(filepath))
    savefig(plot_obj, filepath)
    println("Saved $(filepath)")
end


In [ ]:
# save
"""
    save_record(filepath, key_name, data) -> Nothing

Save `data` under `key_name` in the JLD2 results file, overwriting any existing entry.

# Arguments
- `filepath`: Path to the `.jld2` results file.
- `key_name`: Identifier for the data within the file.
- `data`: Experiment result object to store.
"""
function save_record(filepath, key_name, data)
    jldopen(filepath, "a+") do file
        if haskey(file, key_name)
            delete!(file, key_name)
        end
        file[key_name] = data
    end
end

"""
    delete_record(filepath, key_name) -> Nothing

Delete `key_name` from the JLD2 results file, if present.

# Arguments
- `filepath`: Path to the `.jld2` results file.
- `key_name`: Identifier for the data within the file.
"""
function delete_record(filepath, key_name)
    jldopen(filepath, "a+") do file
        if haskey(file, key_name)
            delete!(file, key_name)
        end
    end
end

filepath = joinpath(ROOT_DIR, "notebooks", "data", "experiments_results.jld2")

# Experiment 1

Each entry in `EXPERIMENTS` is a variation of the default Experiment 1 setup. The result is saved in
`experiments_results.jld2` under the experiment name:

| Name | Description |
|---|---|
| `exp_1_standard` | Default parameters (sign activation, synchronous, Wagner selection, stable initial population, `c=1.0`). |
| `exp_1_regime_shift` | {0,1} regime: step activation, `activation_regime=(0,1)`, and an alternative distance. |
| `exp_1_c=<c>` | Initial network density `c` in `[0.9, 0.5, 0.1, 0.05]`. |
| `exp_1_pop_type_<type>` | Initial population type: `random`, `optimal_clones`, `nonoptimal_clones`, `ensemble_sample` (`stable` is the standard). |
| `exp_1_selection_pressure_s=<s>` | Selection pressure `s` in `[0.0, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0]` (`10.0` is the standard). unstable_fitness = `exp(-10.0)` is fixed|
| `exp_1_selection_roulette` | Roulette selection instead of Wagner selection. |
| `exp_1_development_asynchronous` | Asynchronous (one gene at a time) development. |

Naming scheme: `exp_1_<factor>_<value>` (or `<param>=<value>` for numeric parameters).

**Smoke test:** set `TRIALS`, `GENERATIONS`, `POP_SIZE` to small values and keep `SAVE_PREFIX = "test_"`
so nothing real is overwritten (delete the `test_*` keys afterwards with `delete_record`).
Use `ONLY` to run a subset.

In [ ]:
# # Example use case - standard parameters

simulation_params = SimulationParameters()
config = Experiment1Config(simulation_parameters=simulation_params)
# results = run_experiment_1(config)

# # save data
# save_record(filepath, "exp_1_standard", results)

# load data
key_name = "exp_1_standard"
exp_1_result = load(joinpath(ROOT_DIR, "notebooks", "data", "experiments_results.jld2"), key_name)

In [ ]:
# ---- Run settings (shrink these to smoke-test; use the full values for real runs) ----
TRIALS      = 30
GENERATIONS = 500
POP_SIZE    = 300
SAVE_PREFIX = ""        # e.g. "test_" for smoke tests, so real records are not overwritten

ONLY = String[]
# {0,1}-regime helpers (hamming_distance assumes {-1,1} states)
step_activation(x) = x > 0 ? 1.0 : 0.0
alt_distance(x,y) = mean(abs.(x - y))

# name => keyword arguments overriding the default SimulationParameters
EXPERIMENTS = Pair{String, NamedTuple}[
    "exp_1_standard" => (;),
    "exp_1_regime_shift" => (activation_function = step_activation,
                             activation_regime = (0.0, 1.0),
                             distance_function = alt_distance),
    [("exp_1_c=$c" => (initial_density = c,)) for c in [0.9, 0.5, 0.1, 0.05]]...,
    [("exp_1_pop_type_$t" => (initial_pop_type = t,))
        for t in [:random, :optimal_clones, :nonoptimal_clones, :ensemble_sample]]...,
    [("exp_1_selection_pressure_s=$s" => (selection_pressure = s,))
        for s in [0.0, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0], unstable_fitness = exp(-10.0)]...,
    "exp_1_selection_roulette" => (selection_type = :roulette, unstable_fitness=exp(-10.0)),
    "exp_1_development_asynchronous" => (development_type = :asynchronous,),
]

In [ ]:
"""
    run_and_save_experiment(name, overrides) -> Nothing

Run Experiment 1 with `SimulationParameters` overridden by `overrides` and the global
`TRIALS`/`GENERATIONS`/`POP_SIZE`, then save the result under `SAVE_PREFIX * name`.
"""
function run_and_save_experiment(name, overrides)
    params = SimulationParameters(; generations = GENERATIONS, pop_size = POP_SIZE, overrides...)
    config = Experiment1Config(
        simulation_parameters = params,
        trials = TRIALS,
        sample_points = [1, GENERATIONS],
        sample_sizes = min(30, POP_SIZE),
    )
    save_record(filepath, SAVE_PREFIX * name, run_experiment_1(config))
    return nothing
end


In [ ]:
# Run all selected experiments; a failure in one does not stop the rest
failed = String[]
for (name, overrides) in EXPERIMENTS
    !isempty(ONLY) && !(name in ONLY) && continue
    println("=== Running $(SAVE_PREFIX * name) (trials=$TRIALS, generations=$GENERATIONS, pop_size=$POP_SIZE) ===")
    try
        @time run_and_save_experiment(name, overrides)
    catch err
        @error "Experiment $name failed" exception = (err, catch_backtrace())
        push!(failed, name)
    end
end
println(isempty(failed) ? "All experiments finished." : "Failed experiments: $failed")


# Experiment 1: Time-Series Metrics

In [ ]:
function add_variance_colorbar!(plt, scenarios, colorscheme; sp::Int=1)
    isempty(scenarios) && return
    min_v = minimum(s.variance for s in scenarios)
    max_v = maximum(s.variance for s in scenarios)
    plot!(plt, 0:1, fill(NaN, 2);
        subplot=sp,                  # NEW
        zcolor=[min_v, max_v], c=colorscheme, label="",
        colorbar=true, colorbar_title="Noise Variance " * L"(\theta)",
        grid=false, colorbar_titlefont=12, colorbar_width=4)
end

function add_zoom_inset!(
    plt,
    result::Experiment1Result,
    metric_key::Symbol;
    xzoom::Tuple{<:Real,<:Real},
    yzoom::Union{Nothing,Tuple{<:Real,<:Real}}=nothing,
    sp::Int=1,                       # parent subplot index
    inset_bbox=(0.70, 0.18, 0.28, 0.72),
    inset_title::AbstractString="Zoom near steady state",
    colorscheme=cgrad(:blues),
    init_gen::Int=1,
    ci_mult::Real=1.96,
    rect_lw::Real=2,
    rect_lc=:black,
    rect_ls=:dash,
)
    metric = getfield(result, metric_key)
    metric_avgs = metric[:avg]
    metric_sems = metric[:sem] ./ 1.96
    max_gens = size(metric_avgs, 2)
    scenarios = result.scenarios

    # 1) Zoom region (this is the block that defines x1, x2, y1, y2)
    x1, x2 = xzoom
    if yzoom === nothing
        g1 = max(init_gen, Int(floor(x1)))
        g2 = min(max_gens, Int(ceil(x2)))
        ys = vec(metric_avgs[:, g1:g2])
        ypad = 0.05 * (maximum(ys) - minimum(ys) + eps())
        y1 = minimum(ys) - ypad
        y2 = maximum(ys) + ypad
    else
        y1, y2 = yzoom
    end

    # Rectangle on the parent panel
    rect = Shape([x1, x2, x2, x1], [y1, y1, y2, y2])
    plot!(plt, rect; subplot=sp, c=:transparent, lc=rect_lc, lw=rect_lw, ls=rect_ls, label="")

    # 2) Create the inset; it is appended after all existing subplots
    inset_sp = length(plt.subplots) + 1
    plot!(plt; inset=(sp, bbox(inset_bbox...)))

    # 3) Re-plot the series into the inset
    gens = init_gen:max_gens

    plot!(
        plt,
        1:max_gens,
        metric_avgs[1, 1:max_gens];
        ribbon=metric_sems[1, 1:max_gens] .* ci_mult,
        color=:black,
        linestyle=:dot,
        linewidth=2,
        label="",
        subplot=inset_sp,
        xlims=(x1, x2),
        ylims=(y1, y2),
        title=inset_title,
        grid=false,
        majorgrid=false,
        minorgrid=false,
        legend=false,
        tickfontsize=10,
    )

    noisy_scenarios = scenarios[2:end]
    if !isempty(noisy_scenarios)
        min_var = minimum(s.variance for s in noisy_scenarios)
        max_var = maximum(s.variance for s in noisy_scenarios)
        denom = max(max_var - min_var, eps())

        for (offset, scenario) in enumerate(noisy_scenarios)
            rel_idx = offset + 1
            norm_v = (scenario.variance - min_var) / denom
            line_color = get(colorscheme, clamp(norm_v, 0.0, 1.0))

            plot!(
                plt,
                gens,
                metric_avgs[rel_idx, init_gen:max_gens];
                ribbon=metric_sems[rel_idx, init_gen:max_gens] .* ci_mult,
                color=line_color,
                linewidth=2,
                label="",
                subplot=inset_sp,
            )
        end
    end

    return plt
end

function plot_metric_many_trials(
    result::Experiment1Result,
    metric_key::Symbol;
    plt=nothing,
    sp::Int=1,
    xlabel::AbstractString="Generation",
    ylabel::AbstractString="",
    colorscheme=cgrad(:blues),
    init_gen::Int=1,
    title::AbstractString="",
    ci_mult::Real=1.96,
    outfile::Union{Nothing,AbstractString}=nothing,
    variance_colorbar::Bool=true,
)
    metric = getfield(result, metric_key)
    metric_avgs = metric[:avg]
    metric_sems = metric[:sem]
    max_gens = size(metric_avgs, 2)
    scenarios = result.scenarios
    gens = init_gen:max_gens

    standalone = plt === nothing
    plt = standalone ? plot(size=(600, 400), margins=5Plots.mm, dpi=600) : plt

    # Baseline (scenario 1).
    plot!(
        plt,
        1:max_gens,
        metric_avgs[1, 1:max_gens];
        subplot=sp,
        ribbon=metric_sems[1, 1:max_gens],
        color=:black,
        linestyle=:dot,
        linewidth=2,
        label=scenarios[1].label,
        xlabel=xlabel,
        ylabel=ylabel,
        title=title,
        grid=false,
        majorgrid=false,
        minorgrid=false,
        legendfontsize=12,
        tickfontsize=12,
        guidefontsize=12,
        legend=:best
    )

    # Noisy scenarios (2:end), colored by variance.
    noisy_scenarios = scenarios[2:end]
    if !isempty(noisy_scenarios)
        min_var = minimum(s.variance for s in noisy_scenarios)
        max_var = maximum(s.variance for s in noisy_scenarios)
        denom = max(max_var - min_var, eps())

        for (offset, scenario) in enumerate(noisy_scenarios)
            rel_idx = offset + 1
            norm_v = (scenario.variance - min_var) / denom
            line_color = get(colorscheme, clamp(norm_v, 0.0, 1.0))

            plot!(
                plt,
                gens,
                metric_avgs[rel_idx, init_gen:max_gens];
                subplot=sp,
                ribbon=metric_sems[rel_idx, init_gen:max_gens] .* ci_mult,
                color=line_color,
                linewidth=2,
                label=(rel_idx == 2 ? "Noisy" : ""),
            )
        end
        if variance_colorbar
            add_variance_colorbar!(plt, noisy_scenarios, colorscheme; sp=sp)
        end
    end

    if outfile !== nothing && standalone
        save_plot(plt, outfile)
    end

    return plt
end

### Figure 3: Fitness and Path Length

In [ ]:
key_name = "exp_1_standard"
exp_1_result = load(joinpath(ROOT_DIR, "notebooks", "data", "experiments_results.jld2"), key_name)

key_tag = "exp_1_standard"
noisy = exp_1_result.scenarios[2:end]

p = plot(layout=(1, 2), size=(1400, 400), margins=8Plots.mm, dpi=600)

# ---------- Panel A: fitness ----------
plot_metric_many_trials(exp_1_result, :fitness; plt=p, sp=1,
    ylabel="Average Population Fitness", colorscheme=cgrad(:blues),
    variance_colorbar=false)
add_zoom_inset!(p, exp_1_result, :fitness; sp=1,
    xzoom=(450, 500), yzoom=(0.89, 1.00),
    inset_bbox=(0.30, 0.25, 0.382, 0.35), inset_title="")      # becomes subplot 3
add_variance_colorbar!(p, noisy, cgrad(:blues); sp=1)

for (xs, ys) in (([228, 450], [0.77, 1.0]), ([228, 450], [0.405, 0.92]),
    ([460, 498], [0.77, 1.0]), ([460, 500], [0.41, 0.92]))
    plot!(p, xs, ys; subplot=1, linewidth=2, color=:black, alpha=0.2, label="")
end

# ---------- Panel B: path length ----------
plot_metric_many_trials(exp_1_result, :path_length; plt=p, sp=2,
    ylabel="Average Path Length", colorscheme=cgrad(:greens),
    variance_colorbar=false)
add_zoom_inset!(p, exp_1_result, :path_length; sp=2,
    xzoom=(450, 500), yzoom=(1.8, 3.5),
    inset_bbox=(0.4, 0.05, 0.30, 0.40), inset_title="",
    colorscheme=cgrad(:greens))                                # becomes subplot 4
add_variance_colorbar!(p, noisy, cgrad(:greens); sp=2)

plot!(p; subplot=2, legend=(0.175, 0.9))
for (xs, ys) in (([270, 450], [6.8, 3.5]), ([270, 450], [4.65, 1.8]),
    ([480, 498], [6.8, 3.5]), ([480, 498], [4.65, 1.8]))
    plot!(p, xs, ys; subplot=2, linewidth=2, color=:black, alpha=0.3, label="")
end
initial_path_length = exp_1_result.path_length[:avg][1]
scatter!(p, [0], [initial_path_length]; subplot=2,
    label="Gen 1 = " * @sprintf("%.1f", initial_path_length),
    color=:black, markershape=:rect, markersize=5)

# ---------- Panel labels ----------
plot!(p; subplot=1, title="A", titlelocation=:left, titlefontsize=12)
plot!(p; subplot=2, title="B", titlelocation=:left, titlefontsize=12)

save_plot(p, "exp_1/$key_tag/fit_and_path.png")
display(p)

In [ ]:
plot_avg_completion = plot_metric_many_trials(exp_1_result, :completion;
    ylabel="Average Completion", outfile="exp_1/$key_tag/completion_evolution.png", 
    colorscheme=cgrad(:Purples_8))
display(plot_avg_completion)
plot_avg_alignment = plot_metric_many_trials(exp_1_result, :alignment;
    ylabel="Average Alignment " * L" ⟨A (𝐖) ⟩",
    outfile="exp_1/$key_tag/alignment_evolution.png", colorscheme=cgrad(:reds))
display(plot_avg_alignment)


### Figure 4: Alignment Analysis


In [ ]:
"""
    alignment_scatter_plot(result::Experiment1Result) -> Plots.Plot

Create the final-generation alignment vs. noise scatter plot with a broken x-axis.

# Arguments
- `result::Experiment1Result`: Output of `run_experiment_1`.

# Returns
- The scatter plot object.

# Notes
- Reads the last generation of `result.alignment[:avg]`/`[:sem]`, and noise levels from
    each scenario's `variance` field (there is no separate config attached to the result).
"""
function alignment_scatter_plot(result::Union{ExperimentHandler.Experiment1Result, Experiment1Result})
    final_alignments = result.final_avg_alignment_score
    n_scenarios, n_trials = size(final_alignments)
    thetas = vcat(0.0, result.config.thetas)
    @assert length(thetas) == n_scenarios

    y_flat = vec(final_alignments)                  # scenario varies fastest, then trial
    theta_flat = repeat(thetas, outer=n_trials)      # cycles thetas once per trial — matches y_flat's order

    p1 = scatter(
        sqrt.(theta_flat),
        color=:brown3,
        y_flat,
        alpha=0.3,
        markersize=4,
        markerstrokeopacity=0,
        legend=false
    )
    
    # Per-scenario median and empirical 95% interval (2.5%-97.5% percentiles) from final_alignments
    medians = [median(final_alignments[i, :]) for i in 1:n_scenarios]
    lowers = [quantile(final_alignments[i, :], 0.05) for i in 1:n_scenarios]
    uppers = [quantile(final_alignments[i, :], 0.95) for i in 1:n_scenarios]

    x_vals = [0.0; [sqrt(s.variance) for s in result.scenarios[2:end]]]

    xmin, xmax = extrema(x_vals)
    xpad = (xmax - xmin) * 0.1
    shared_xlims = (xmin - xpad, xmax + xpad)

    scatter!(p1, x_vals,
        medians;
        yerror=(medians .- lowers, uppers .- medians),  # asymmetric: percentile interval isn't symmetric in general
        color=:black,
        ylabel="Final Average Alignment " * L" ⟨A (𝐖)⟩",
        legend=false,
        markersize=8,
        grid=false,
        minorgrid=false,
        xlims=shared_xlims,
        xlabel="Noise level " * L"(\sqrt{\theta})",
        markerstrokewidth=1.5,
        markershape=:x,
        errorbarlinewidth=0.5,
        errorbarcapsize=2,
        dpi=600,
    )

    return p1
end

In [ ]:
# Figure 4: alignment time series + final alignment scatter.
scatter_final_alignment = alignment_scatter_plot(exp_1_result)

fig4_combined = plot(plot_avg_alignment, scatter_final_alignment;
    layout=(1, 2), size=(1200, 400),
    left_margin=12Plots.mm,
    bottom_margin=6Plots.mm, legendfontsize=12, guidefont=12, tickfont=12)

plot!(fig4_combined[1], title="A", titlelocation=:left, titlefontsize=12, ylim=(-0.05,1.0))
plot!(fig4_combined[2], title="B", titlelocation=:left, titlefontsize=12, ylim=(-0.05, 1.0))

save_plot(fig4_combined, "exp_1/$key_tag/alignment_combined.png")
fig4_combined

## Mutational Robustness Scatters


In [ ]:
"""Sanitize a string for filenames by replacing non-alphanumerics."""
sanitize_filename(name::String) = replace(lowercase(name), r"[^a-z0-9]+" => "_")

"""
    mutational_scatter_plot(xvals, yvals; xlabel, ylabel, title, xrange=(-0.05, 1.05), yrange=(-0.05, 1.05))

Create a single mutational robustness scatter plot with a diagonal reference line.
"""
function mutational_scatter_plot(
    xvals,
    yvals;
    xlabel::String,
    ylabel::String,
    title::String,
    xrange::Tuple{Float64,Float64}=(-0.05, 1.05),
    yrange::Tuple{Float64,Float64}=(-0.05, 1.05)
)
    plt = scatter(
        xvals,
        yvals;
        xlabel=xlabel,
        ylabel=ylabel,
        color=:darkorange1,
        title=title,
        label="",
        legend=false,
        markersize=6,
        markerstrokewidth=0.6,
        grid=false,
        majorgrid=false,
        minorgrid=false,
        left_margin=15Plots.mm,
        bottom_margin=10Plots.mm,
        right_margin=5Plots.mm,
        xlims=xrange,
        ylims=yrange,
        dpi=600,
    )
    plot!(
        [xrange[1], xrange[2]],
        [yrange[1], yrange[2]];
        color=:black,
        linestyle=:dash,
        label="",
        grid=false,
    )
    return plt
end

"""Normalize theta in labels to LaTeX-friendly markup."""
normalize_theta(s::String) = replace(replace(s, string(Char(0x03B8)) => L"\theta"), string(Char(0x00CE), Char(0x00B8)) => L"\theta")

"""
    metric_scatter_limits(xvals, yvals; symmetric=false, center=0.0)

Compute dynamic scatter limits from two vectors with a small padding.
For signed metrics, use `symmetric=true` to keep centered axes.
"""
function metric_scatter_limits(xvals, yvals; symmetric::Bool=false, center::Float64=0.0)
    x = Float64.(collect(xvals))
    y = Float64.(collect(yvals))
    vals = vcat(x, y)

    if symmetric
        radius = maximum(abs.(vals .- center))
        pad = max(0.08 * radius, 1e-3)
        return (center - radius - pad, center + radius + pad)
    end

    lo = minimum(vals)
    hi = maximum(vals)
    span = hi - lo
    pad = span > 0 ? 0.08 * span : max(0.1 * max(abs(lo), abs(hi), 1.0), 1e-3)
    return (lo - pad, hi + pad)
end

"""
    handle_mutational_robustness_scatters(result::Experiment1Result) -> Vector{Any}

Generate per-scenario mutational robustness scatters (initial vs. final sample point) and
save one combined figure per scenario to `outdir`.

# Notes
- "Initial" and "final" are the first and last entries of `result.mut_rob`'s sample-point
    axis, i.e. `Experiment1Config.sample_points[1]` and `[end]` — make sure that vector is
    ordered from earliest to latest generation.
"""
function handle_mutational_robustness_scatters(result::Experiment1Result; tag::String="",
        outdir::AbstractString=ROBUSTNESS_FIG_DIR)
    stab_shift_init = result.mut_rob.stable_expression_shift[:, :, 1]
    stab_shift_final = result.mut_rob.stable_expression_shift[:, :, end]
    stab_var_init = result.mut_rob.stable_expression_var[:, :, 1]
    stab_var_final = result.mut_rob.stable_expression_var[:, :, end]
    unstab_shift_init = result.mut_rob.unstable_prob_shift[:, :, 1]
    unstab_shift_final = result.mut_rob.unstable_prob_shift[:, :, end]
    unstab_var_init = result.mut_rob.unstable_prob_var[:, :, 1]
    unstab_var_final = result.mut_rob.unstable_prob_var[:, :, end]

    n = length(result.scenarios)
    all_plots = Vector{Any}(undef, n)

    for idx in eachindex(result.scenarios)
        scenario = result.scenarios[idx]
        label_token = sanitize_filename(scenario.label)
        lab = normalize_theta(scenario.label)

        shift_limits = metric_scatter_limits(stab_shift_init[idx, :], stab_shift_final[idx, :])
        var_limits = metric_scatter_limits(stab_var_init[idx, :], stab_var_final[idx, :])
        unstab_shift_limits = metric_scatter_limits(
            unstab_shift_init[idx, :],
            unstab_shift_final[idx, :];
            symmetric=true,
            center=0.0,
        )
        unstab_var_limits = metric_scatter_limits(unstab_var_init[idx, :], unstab_var_final[idx, :])

        p1 = mutational_scatter_plot(
            stab_shift_init[idx, :],
            stab_shift_final[idx, :];
            xlabel="Before Evolution",
            ylabel="After Evolution",
            title="Stable Expression Shift $lab " * L"\langle \Delta \bar{v} \,( 𝐖 ) \rangle",
            xrange=shift_limits,
            yrange=shift_limits,
        )

        p2 = mutational_scatter_plot(
            stab_var_init[idx, :],
            stab_var_final[idx, :];
            xlabel="Before Evolution",
            ylabel="After Evolution",
            title="Stable Expression Variance $lab " * L"\langle \sigma^2( 𝐖 ) \rangle",
            xrange=var_limits,
            yrange=var_limits,
        )

        p3 = mutational_scatter_plot(
            unstab_shift_init[idx, :],
            unstab_shift_final[idx, :];
            xlabel="Before Evolution",
            ylabel="After Evolution",
            title="Unstable Probability Shift $lab " * L"\langle p_u( 𝐖 ) \rangle",
            xrange=unstab_shift_limits,
            yrange=unstab_shift_limits,
        )

        p4 = mutational_scatter_plot(
            unstab_var_init[idx, :],
            unstab_var_final[idx, :];
            xlabel="Before Evolution",
            ylabel="After Evolution",
            title="Unstable Probability Variance $lab " * L"\langle \rho^2(𝐖) \rangle",
            xrange=unstab_var_limits,
            yrange=unstab_var_limits,
        )

        combined = plot(p1, p2, p3, p4; layout=(2, 2), size=(1150, 800), grid=false)
        savefig(combined, joinpath(outdir, "robustness_$(label_token)" * (isempty(tag) ? "" : "_$tag") * ".png"))
        all_plots[idx] = combined
    end

    return all_plots
end

In [ ]:
# Figure set: mutational robustness scatter grids (each is saved internally, one file per scenario).
all_plots = handle_mutational_robustness_scatters(exp_1_result)
foreach(display, all_plots)

## Mutational Robustness vs Noise


In [ ]:
"""
    initial_vs_final_mutational_robustness(result::Experiment1Result) -> NTuple{4,Plots.Plot}

Compare initial vs. final (first vs. last sample point) robustness across noise levels,
plotting both values (mean ± 95% CI over trials) rather than their difference. See the note on `handle_mutational_robustness_scatters` about what "initial"/"final"
mean here.
"""
function initial_vs_final_mutational_robustness(result::Experiment1Result;
        outdir::AbstractString=ROBUSTNESS_FIG_DIR)
    stab_shift_init = result.mut_rob.stable_expression_shift[:, :, 1]
    stab_shift_final = result.mut_rob.stable_expression_shift[:, :, end]
    stab_var_init = result.mut_rob.stable_expression_var[:, :, 1]
    stab_var_final = result.mut_rob.stable_expression_var[:, :, end]
    unstab_shift_init = result.mut_rob.unstable_prob_shift[:, :, 1]
    unstab_shift_final = result.mut_rob.unstable_prob_shift[:, :, end]
    unstab_var_init = result.mut_rob.unstable_prob_var[:, :, 1]
    unstab_var_final = result.mut_rob.unstable_prob_var[:, :, end]

    theta_vals = [scenario.variance for scenario in result.scenarios]
    x = sqrt.(theta_vals)

    function summarize_metric(metric)
        means = [mean(metric[idx, :]) for idx in eachindex(result.scenarios)]
        cis = [1.96 * std(metric[idx, :]) / sqrt(size(metric, 2))
               for idx in eachindex(result.scenarios)]
        return means, cis
    end

    common_kwargs = (
        grid=false,
        minorgrid=false,
        markersize=7,
        markerstrokewidth=1,
        errorbarlinewidth=2,
        errorbarcapsize=4,
        linecolor=:black,
        dpi=600,
    )

    # Scatter "before evolution" (hollow gray squares) and "after evolution" (orange circles)
    # of the same metric against noise level. Legend is only drawn in the first panel.
    function before_after_plot(initial_metric, final_metric, ylabel; legend=false)
        init_mean, init_ci = summarize_metric(initial_metric)
        final_mean, final_ci = summarize_metric(final_metric)
        plt = scatter(
            x,
            init_mean;
            yerror=init_ci,
            xlabel="Noise level " * L"(\sqrt{\theta})",
            ylabel=ylabel,
            label="Before evolution",
            marker=:square,
            color=:white,
            markerstrokecolor=:gray40,
            legend=legend ? :best : false,
            common_kwargs...,
        )
        scatter!(
            plt,
            x,
            final_mean;
            yerror=final_ci,
            label="After evolution",
            marker=:circle,
            color=:darkorange1,
            common_kwargs...,
        )
        return plt
    end

    stab_shift_vs_theta = before_after_plot(stab_shift_init, stab_shift_final,
        "Stable Expression Shift " * L"\langle \Delta \bar{v} \, (𝐖) \rangle"; legend=true)
    stab_var_vs_theta = before_after_plot(stab_var_init, stab_var_final,
        "Stable Phenotype Variance " * L"\langle \sigma^2 (𝐖)\rangle")
    unstab_shift_vs_theta = before_after_plot(unstab_shift_init, unstab_shift_final,
        "Instability Shift " * L"\langle p_u (𝐖) \rangle")
    unstab_var_vs_theta = before_after_plot(unstab_var_init, unstab_var_final,
        "Instability Variance " * L"\langle \rho^2 (𝐖) \rangle")

    savefig(stab_shift_vs_theta, joinpath(outdir, "stable_expression_shift_vs_theta.png"))
    savefig(stab_var_vs_theta, joinpath(outdir, "stable_expression_variance_vs_theta.png"))
    savefig(unstab_shift_vs_theta, joinpath(outdir, "unstable_probability_shift_vs_theta.png"))
    savefig(unstab_var_vs_theta, joinpath(outdir, "unstable_probability_variance_vs_theta.png"))

    return stab_shift_vs_theta, stab_var_vs_theta, unstab_shift_vs_theta, unstab_var_vs_theta
end

In [ ]:
stab_shift_vs_theta, stab_var_vs_theta, unstab_shift_vs_theta, unstab_var_vs_theta = initial_vs_final_mutational_robustness(exp_1_result)

legend_panel = plot(
    framestyle=:none,
    grid=false,
    xticks=false,
    yticks=false,
    xaxis=false,
    yaxis=false,
    legend=:left,
    legend_columns=2,
    legendfontsize=12,
    foreground_color_legend=nothing,   # no legend border
    background_color_legend=nothing,
)

scatter!(legend_panel, [NaN], [NaN];
    label="Before evolution",
    marker=:square,
    markersize=7,
    markerstrokewidth=1,
    color=:white,
    markerstrokecolor=:gray40,
)

scatter!(legend_panel, [NaN], [NaN];
    label="After evolution",
    marker=:circle,
    markersize=7,
    markerstrokewidth=1,
    color=:darkorange1,
    markerstrokecolor=:black,
)

lay = @layout [
    legend{0.03h} # legend strip (7% of height)
    (2, 2)    # 4 data panels
]

robustness_deltas = plot(
    legend_panel,
    stab_shift_vs_theta,
    stab_var_vs_theta,
    unstab_shift_vs_theta,
    unstab_var_vs_theta;
    layout=lay,
    size=(1150, 820),
    left_margin=12Plots.mm,
    bottom_margin=5Plots.mm,
    guidefont=12,
    tickfont=12,
    dpi=600,
)

plot!(robustness_deltas[2], title="A", titlelocation=:left, titlefontsize=12, legend=false)
plot!(robustness_deltas[3], title="B", titlelocation=:left, titlefontsize=12)
plot!(robustness_deltas[4], title="C", titlelocation=:left, titlefontsize=12)
plot!(robustness_deltas[5], title="D", titlelocation=:left, titlefontsize=12)

save_plot(robustness_deltas, "exp_1/$key_tag/scatterplots_mutational_robustness/deltas_vs_theta.png")
display(robustness_deltas)

## Generate All Experiment 1 Figures

Loops over every saved `exp_1_*` dataset and writes all Experiment 1 figures (without zoom insets) to `figures/exp_1/<dataset>/`. 
Run the experiment-setup cell first so functions stored in the configs (e.g. `step_activation`) exist before loading.


In [ ]:
# Generates every Experiment 1 figure (no zoom insets) for each saved dataset, one folder per
# dataset: figures/exp_1/<dataset_key>/
#   fit_and_path.png, completion_evolution.png, alignment_evolution.png, alignment_combined.png,
#   scatterplots_mutational_robustness/{robustness_<scenario>.png, deltas_vs_theta.png, ...}

# Datasets to plot: `nothing` = every `exp_1_*` key in the results file, or e.g. ["exp_1_standard"]
DATASETS = nothing

"""
    fit_and_path_figure(result::Experiment1Result) -> Plots.Plot

Fitness (A) and path length (B) over generations, without zoom insets. The black square on
panel B marks the average path length at generation 1.
"""
function fit_and_path_figure(result::Experiment1Result)
    noisy = result.scenarios[2:end]
    p = plot(layout=(1, 2), size=(1400, 400), margins=8Plots.mm, dpi=600)

    plot_metric_many_trials(result, :fitness; plt=p, sp=1,
        ylabel="Average Population Fitness", colorscheme=cgrad(:blues),
        variance_colorbar=false)
    add_variance_colorbar!(p, noisy, cgrad(:blues); sp=1)

    plot_metric_many_trials(result, :path_length; plt=p, sp=2,
        ylabel="Average Path Length", colorscheme=cgrad(:greens),
        variance_colorbar=false)
    add_variance_colorbar!(p, noisy, cgrad(:greens); sp=2)

    initial_path_length = result.path_length[:avg][1]
    scatter!(p, [0], [initial_path_length]; subplot=2,
        label="Generation 1: " * @sprintf("%.2f", initial_path_length),
        color=:black, markershape=:rect, markersize=5)

    plot!(p; subplot=1, title="A", titlelocation=:left, titlefontsize=16, legend=:topright)
    plot!(p; subplot=2, title="B", titlelocation=:left, titlefontsize=16, legend=:topright)
    return p
end

"""
    generate_experiment_1_figures(key, result) -> Nothing

Save all Experiment 1 figures for `result` under `figures/exp_1/<key>/`.
"""
function generate_experiment_1_figures(key::String, result::Experiment1Result)
    robustness_dir = joinpath(FIGURES_DIR, "exp_1", key, "scatterplots_mutational_robustness")
    mkpath(robustness_dir)

    # Fitness and path length
    fit_and_path_fig = fit_and_path_figure(result)
    plot!(fit_and_path_fig[1], legend=:bottomright)
    save_plot(fit_and_path_fig, "exp_1/$key/fit_and_path.png")

    # Completion and alignment time series, plus the alignment time series + final scatter
    plot_metric_many_trials(result, :completion; ylabel="Average Completion",
        outfile="exp_1/$key/completion_evolution.png", colorscheme=cgrad(:Purples_8))
    alignment_series = plot_metric_many_trials(result, :alignment;
        ylabel="Average Alignment " * L" ⟨A (𝐖) ⟩",
        outfile="exp_1/$key/alignment_evolution.png", colorscheme=cgrad(:reds))
    alignment_combined = plot(alignment_series, alignment_scatter_plot(result);
        layout=(1, 2), size=(1200, 400), left_margin=12Plots.mm, bottom_margin=6Plots.mm,
        legendfontsize=12, guidefont=12, tickfont=12)
    plot!(alignment_combined[1], title="A", titlelocation=:left, titlefontsize=12)
    plot!(alignment_combined[2], title="B", titlelocation=:left, titlefontsize=12)
    save_plot(alignment_combined, "exp_1/$key/alignment_combined.png")

    # Mutational robustness: before-vs-after scatter per noise scenario
    handle_mutational_robustness_scatters(result; outdir=robustness_dir)

    # Mutational robustness vs noise (before and after evolution)
    panels = initial_vs_final_mutational_robustness(result; outdir=robustness_dir)
    robustness_change = plot(panels...; layout=(2, 2), size=(1150, 780),
        left_margin=12Plots.mm, bottom_margin=5Plots.mm, legendfontsize=12, guidefont=12,
        tickfont=12, dpi=600)
    for (i, letter) in enumerate("ABCD")
        plot!(robustness_change[i], title=string(letter), titlelocation=:left, titlefontsize=12, legend=false)
    end
    save_plot(robustness_change, "exp_1/$key/scatterplots_mutational_robustness/deltas_vs_theta.png")
    return nothing
end

dataset_keys = something(DATASETS,
    sort(filter(k -> startswith(k, "exp_1_"), collect(jldopen(keys, filepath, "r")))))

failed_figures = String[]
for key in dataset_keys
    println("=== Generating figures for $key ===")
    try
        result = jldopen(f -> f[key], filepath, "r")
        @time generate_experiment_1_figures(key, result)
    catch err
        @error "Figures for $key failed" exception = (err, catch_backtrace())
        push!(failed_figures, key)
    end
    GC.gc()  # results hold large matrix samples; free them before loading the next one
end
println(isempty(failed_figures) ? "All figures generated." : "Failed datasets: $failed_figures")


# Experiment 2

In [ ]:
simulation_parameters = SimulationParameters()

exp_2_config = Experiment2Config(
    simulation_parameters=simulation_parameters,
    trials=30,
    max_loop_size=5,
    sample_size=100,
    high_noise_dist=Gamma(1.0/8.0, 8.0),
)

# exp_2_result = run_experiment_2(exp_2_config)

# # save
# key_2_name = """exp_2_standard"""
# save_record(filepath, key_2_name, exp_2_result)

# load
key_2_name = """exp_2_standard"""
exp_2_result = load(joinpath(ROOT_DIR, "notebooks", "data", "experiments_results.jld2"), key_2_name)

## Experiment 2: Motif Summary Plots


In [ ]:
# Motif summary helpers
# One shape per condition (circle / triangle / star), one color per motif class, so the
# same marker always means the same population across both figures.
const CONDITION_SHAPES = Dict("Non-Evolved" => :circle, "Moderate Noise" => :utriangle,
    "High Noise" => :star5)
const FFL_COLORS = (coherent="#348ABD", incoherent="#bf2b54")
const FBCK_COLORS = (positive="#5cbf8a", negative="#e94e77")
const CI_MULT = 1.96
const X_SPACING = 1.0  # distance between consecutive motif types (1 = tightly packed)

"""
    mean_se(data, key) -> Tuple{Float64,Float64}

Mean and standard error of `data[key]` across trials, or `(0.0, 0.0)` if `key` is absent.
"""
function mean_se(data::AbstractDict, key)
    haskey(data, key) || return (0.0, 0.0)
    v = data[key]
    return (mean(v), std(v) / sqrt(length(v)))
end

"""
    condition_offsets(n, width) -> Vector{Float64}

Horizontal offsets that spread `n` conditions symmetrically around each x tick.
"""
condition_offsets(n::Int, width::Real) = n == 1 ? [0.0] : collect(range(-width, width; length=n))

"""
    add_shape_legend!(plt, labels, class_entries)

Add NaN-only legend entries: gray markers for each condition shape, then one colored circle
per class in `class_entries` (a vector of `label => color`).
"""
function add_shape_legend!(plt, labels, class_entries, shape=:square)
    for lab in labels
        scatter!(plt, [NaN], [NaN]; label=lab, marker=CONDITION_SHAPES[lab], markersize=7,
            color=:gray, markerstrokecolor=:black, markerstrokewidth=1.5)
    end
    for (lab, col) in class_entries
        scatter!(plt, [NaN], [NaN]; label=lab, marker=shape, markersize=7,
            color=col, markerstrokecolor=:black, markerstrokewidth=1.5)
    end
    return plt
end

"""
    ffl_tick_labels(motif_keys) -> Vector{String}

Short x tick labels for FFL types, taken from the parenthesized code in each name
(e.g. "Coherent Type 1 (C1)" -> "(C1)").
"""
ffl_tick_labels(motif_keys) = [(m = match(r"\([CI]\d\)", k)) === nothing ? k : m.match
                               for k in motif_keys]

"""
    create_ffl_summary_plot(conditions; title="") -> Plots.Plot

Feedforward-loop frequency (`relative` selects the `_relative` fields; mean ± 95% CI) per motif type. `conditions` is a vector of
`(label, stats)` pairs, where `stats` is a `MotifStats`. Coherent types are blue and
incoherent types red; the condition is encoded by marker shape.
X tick labels are blank by default so motif images can be overlaid later; with
`tick_labels=true` they are the short motif names, e.g. (C1), (I4). "Coherent"/"Incoherent"
labels sit below the axis (`label_offset` is a fraction of the y range).
"""
function create_ffl_summary_plot(conditions; title::String="", relative::Bool=true,
        label_offset::Real=0.25, tick_labels::Bool=false)
    motif_keys = sort(collect(keys(first(conditions)[2].ffl)))
    ffl_field = relative ? :ffl_relative : :ffl
    x = X_SPACING .* collect(1:length(motif_keys))
    is_incoherent = occursin.("incoherent", lowercase.(motif_keys))
    groups = [("Coherent", findall(.!is_incoherent), FFL_COLORS.coherent),
              ("Incoherent", findall(is_incoherent), FFL_COLORS.incoherent)]

    summaries = [[mean_se(getfield(stats, ffl_field), k) for k in motif_keys]
                 for (_, stats) in conditions]

    # Fix the y range so the class labels can be placed just below the x axis
    lo = minimum(minimum(first.(s) .- CI_MULT .* last.(s)) for s in summaries)
    hi = maximum(maximum(first.(s) .+ CI_MULT .* last.(s)) for s in summaries)
    pad = 0.05 * (hi - lo + eps())
    ymin, ymax = lo - pad, hi + pad

    plt = plot(; xticks=(x, tick_labels ? ffl_tick_labels(motif_keys) : fill("", length(x))), xlims=(x[1] - X_SPACING * 0.7, x[end] + X_SPACING * 0.7),
        ylims=(ymin, ymax),
        xlabel="", ylabel=relative ? "Relative Frequency" : "Frequency (count)", title=title,
        titlelocation=:left, titlefontsize=12,
        legend=:topright, grid=false, minorgrid=false, minorticks=false,
        left_margin=12Plots.mm, bottom_margin=20Plots.mm, dpi=600)

    offsets = condition_offsets(length(conditions), 0.25 * X_SPACING)
    for (((label, _), summary), off) in zip(zip(conditions, summaries), offsets)
        for (_, idx, col) in groups
            scatter!(plt, x[idx] .+ off, first.(summary[idx]);
                yerror=CI_MULT .* last.(summary[idx]), label=false,
                marker=CONDITION_SHAPES[label], markersize=8, color=col,
                markerstrokecolor=:black, markerstrokewidth=1.5,
                errorbarcolor=:black, errorbarlinewidth=2)
        end
    end

    # Class labels below the x axis, centered under each group of motif types
    label_y = ymin - label_offset * (ymax - ymin)
    for (name, idx, col) in groups
        isempty(idx) && continue
        annotate!(plt, mean(x[idx]), label_y, text(name, "Computer Modern", 14, col))  # font of Plots.theme(:dao))
    end

    add_shape_legend!(plt, first.(conditions),
        ["Coherent" => FFL_COLORS.coherent, "Incoherent" => FFL_COLORS.incoherent])
    return plt
end

"""
    create_feedback_summary_plot(conditions, max_size; title="") -> Plots.Plot

Feedback-loop frequencies (`relative` selects the `_relative` fields; mean ± 95% CI) per loop size, for the positive (reinforcing)
and negative (balancing) loops. `conditions` is a vector of `(label, stats)` pairs.
"""
function create_feedback_summary_plot(conditions, max_size::Int; title::String="", relative::Bool=true)
    sizes = collect(1:max_size)
    plt = plot(; xticks=(sizes, string.(sizes)), xlims=(0.5, max_size + 0.5),
        xlabel="Feedback-loop size", ylabel=relative ? "Relative Frequency" : "Frequency (count)", title=title,
        titlelocation=:left, titlefontsize=12,
        legend=:best, grid=false, minorgrid=false, minorticks=false,
        left_margin=10Plots.mm, right_margin=10Plots.mm, bottom_margin=8Plots.mm, dpi=600)

    offsets = condition_offsets(length(conditions), 0.2)
    for ((label, stats), off) in zip(conditions, offsets)
        for (field, col) in ((relative ? :feedback_reinforcing_relative : :feedback_reinforcing, FBCK_COLORS.positive),
                             (relative ? :feedback_balancing_relative : :feedback_balancing, FBCK_COLORS.negative))
            summary = [mean_se(getfield(stats, field), s) for s in sizes]
            scatter!(plt, sizes .+ off, first.(summary);
                yerror=CI_MULT .* last.(summary), label=false,
                marker=CONDITION_SHAPES[label], markersize=8, color=col,
                markerstrokecolor=:black, markerstrokewidth=1.5,
                errorbarcolor=col, errorbarlinewidth=2)
        end
    end

    add_shape_legend!(plt, first.(conditions),
        ["Positive" => FBCK_COLORS.positive, "Negative" => FBCK_COLORS.negative])
    return plt
end

"""
    handle_motif_summary(conditions, max_loop_size; suptitle="") -> Plots.Plot

Two-panel figure: feedforward-loop frequencies (left) and feedback-loop frequencies
(right) for every `(label, MotifStats)` in `conditions`.

# Arguments
- `conditions`: Vector of `(label, stats)` pairs; labels must be keys of `CONDITION_SHAPES`.
- `max_loop_size::Int`: Largest feedback-loop size to plot (from `Experiment2Config`,
    which is not attached to the result).
"""
function handle_motif_summary(conditions, max_loop_size::Int; suptitle::String="",
        relative::Bool=true, tick_labels::Bool=false)
    return plot(
        create_ffl_summary_plot(conditions; title="A - Feedforward Loops", relative, tick_labels),
        create_feedback_summary_plot(conditions, max_loop_size;
            title="B - Feedback Loops", relative);
        layout=(1, 2), size=(1500, 460), suptitle=suptitle, plot_titlevspan=0.10)
end

"""
    handle_noiseless_vs_random(result::Experiment2Result, max_loop_size::Int) -> Plots.Plot

Motif summary comparing non-evolved (random) and noiseless-evolved populations.
"""
function handle_noiseless_vs_random(result::Union{Experiment2Result,Main.ExperimentHandler.Experiment2Result}, max_loop_size::Int;
        tick_labels::Bool=false)
    return handle_motif_summary(
        ["Non-Evolved" => result.random, "Moderate Noise" => result.noiseless],
        max_loop_size; suptitle="Non-Evolved vs Evolved in Moderate Noise", tick_labels)
end


In [ ]:
using Logging
Logging.disable_logging(Logging.Warn)  # Suppress warnings at or below the Warning level.

fig5_combined = handle_noiseless_vs_random(exp_2_result, exp_2_config.max_loop_size)
plot!(fig5_combined[1])

save_plot(fig5_combined, "exp_2/$key_2_name/noiseless_vs_random.png")
display(fig5_combined)

## Figure 6: High Noise vs Noiseless


In [ ]:
"""
    handle_high_noise_vs_noiseless(result::Experiment2Result, max_loop_size::Int) -> Plots.Plot

Motif summary comparing non-evolved (random), noiseless-evolved and very noisy-evolved
populations.
"""
function handle_high_noise_vs_noiseless(result::Union{Experiment2Result,Main.ExperimentHandler.Experiment2Result}, max_loop_size::Int;
        tick_labels::Bool=false)
    return handle_motif_summary(
        ["Non-Evolved" => result.random, "Moderate Noise" => result.noiseless,
         "High Noise" => result.noisy],
        max_loop_size; suptitle="Non-Evolved vs Evolved in Moderate Noise vs Evolved in High Noise", tick_labels)
end


In [ ]:
# Figure 6: high noise vs noiseless.
fig6_combined = handle_high_noise_vs_noiseless(exp_2_result, exp_2_config.max_loop_size)


save_plot(fig6_combined, "exp_2/$key_2_name/high_noise_vs_noiseless.png")
fig6_combined

## Experiment 2: All Parameter Variations

Runs Experiment 2 with the same variations (`EXPERIMENTS`) and run settings as Experiment 1, saved as `exp_2_<variation>`.

In [ ]:
# Run Experiment 2 for every Experiment 1 variation (same overrides, TRIALS, GENERATIONS, POP_SIZE).
# Results are saved as `exp_2_<variation>`; `ONLY` / `SAVE_PREFIX` from the Experiment 1 settings apply
# (ONLY is matched against the Experiment 1 names).
EXPERIMENTS_2 = [replace(name, "exp_1_" => "exp_2_") => overrides for (name, overrides) in EXPERIMENTS]

"""
    run_and_save_experiment_2(name, overrides) -> Nothing

Run Experiment 2 with `SimulationParameters` overridden by `overrides` and the global
`TRIALS`/`GENERATIONS`/`POP_SIZE`, then save the result under `SAVE_PREFIX * name`.
"""
function run_and_save_experiment_2(name, overrides)
    params = SimulationParameters(; generations = GENERATIONS, pop_size = POP_SIZE, overrides...)
    config = Experiment2Config(
        simulation_parameters = params,
        trials = TRIALS,
        max_loop_size = 5,
        sample_size = 100,
        high_noise_dist = Gamma(1.0/8.0, 8.0),
    )
    save_record(filepath, SAVE_PREFIX * name, run_experiment_2(config))
    return nothing
end

failed_2 = String[]
for (name, overrides) in EXPERIMENTS_2
    !isempty(ONLY) && !(replace(name, "exp_2_" => "exp_1_") in ONLY) && continue
    println("=== Running $(SAVE_PREFIX * name) (trials=$TRIALS, generations=$GENERATIONS, pop_size=$POP_SIZE) ===")
    try
        @time run_and_save_experiment_2(name, overrides)
    catch err
        @error "Experiment $name failed" exception = (err, catch_backtrace())
        push!(failed_2, name)
    end
end
println(isempty(failed_2) ? "All experiment 2 runs finished." : "Failed experiments: $failed_2")

In [ ]:
# Motif figures (Figures 5 and 6) for every saved Experiment 2 variation
EXPERIMENTS_2 = [replace(name, "exp_1_" => "exp_2_") => overrides for (name, overrides) in EXPERIMENTS]

const EXP_2_MAX_LOOP_SIZE = 5
for (name, _) in EXPERIMENTS_2
    !isempty(ONLY) && !(replace(name, "exp_2_" => "exp_1_") in ONLY) && continue
    result = load(filepath, SAVE_PREFIX * name)
    save_plot(handle_noiseless_vs_random(result, EXP_2_MAX_LOOP_SIZE; tick_labels=true),
        joinpath("exp_2", name, "noiseless_vs_random.png"))
    save_plot(handle_high_noise_vs_noiseless(result, EXP_2_MAX_LOOP_SIZE; tick_labels=true),
        joinpath("exp_2", name, "high_noise_vs_noiseless.png"))
end